In [25]:
print("hello! My name is Prakash Chawda")

hello! My name is Prakash Chawda


In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error 
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


In [ ]:
#Load Dataset
customers = pd.read_csv(r"F:\project\Adventure\raw files\Customers.csv" , encoding = 'cp1252')
products  = pd.read_csv(r"F:\project\Adventure\raw files\Products.csv" , encoding = 'cp1252')
product_categories = pd.read_csv(r"F:\project\Adventure\raw files\Product_Categories.csv" , encoding = 'cp1252')
product_subcategories = pd.read_csv(r"F:\project\Adventure\raw files\Product_Subcategories.csv" , encoding = 'cp1252')
returns = pd.read_csv(r"F:\project\Adventure\raw files\Returns.csv" , encoding = 'cp1252')
sales_2015 = pd.read_csv(r"F:\project\Adventure\raw files\Sales_2015.csv" , encoding = 'cp1252')
sales_2016 = pd.read_csv(r"F:\project\Adventure\raw files\Sales_2016.csv" , encoding = 'cp1252')
sales_2017 = pd.read_csv(r"F:\project\Adventure\raw files\Sales_2017.csv" , encoding = 'cp1252')
territories = pd.read_csv(r"F:\project\Adventure\raw files\Territories.csv" , encoding = 'cp1252')

In [ ]:
#Data Cleaning
sales = pd.concat([sales_2015 , sales_2016 , sales_2017] , ignore_index= True)

customers['BirthDate'] = pd.to_datetime(customers['BirthDate'],errors = "coerce")
customers['AnnualIncome'] = customers['AnnualIncome'].str.replace('[\$,]', '' , regex = True).astype(float)
customers['FullName'] = customers[['FirstName' , 'LastName']].agg(' '.join , axis = 1)
pos = customers.columns.get_loc('FirstName')
customers.drop(['FirstName' , 'LastName'], axis = 1 , inplace = True)
col = customers.pop('FullName')
customers.insert(pos , 'FullName' , col)

returns['ReturnDate'] = pd.to_datetime(returns['ReturnDate'] , errors = "coerce")

sales['OrderDate'] = pd.to_datetime(sales['OrderDate'] , errors = "coerce")
sales['StockDate'] = pd.to_datetime(sales['StockDate'] , errors = "coerce")

<>:4: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:4: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
C:\Users\INDIA\AppData\Local\Temp\ipykernel_4216\472533929.py:4: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  customers['AnnualIncome'] = customers['AnnualIncome'].str.replace('[\$,]', '' , regex = True).astype(float)


In [ ]:
#merge Data
df = sales.merge(products[['ProductKey' , 'ProductPrice']] , on = 'ProductKey' , how = 'left')
df['Revenue'] = df['OrderQuantity'] * df['ProductPrice']
returns_df = returns.merge(products[['ProductKey', 'ProductPrice']], on = 'ProductKey' , how = 'left')
returns_df['Return_value'] = returns_df['ReturnQuantity'] * returns_df['ProductPrice']

#Aggregation
sales_agg = df.groupby( by = 'OrderDate' ).agg(func= {'Revenue' : 'sum'}).reset_index()
returns_agg = returns_df.groupby(by = 'ReturnDate' ).agg(func = {'Return_value' : 'sum'}).reset_index()


In [ ]:
#Combine Sales & Returns
final_df = pd.merge(sales_agg , returns_agg , left_on = 'OrderDate',
                      right_on = 'ReturnDate' , how = 'left')
final_df['Return_value'] = final_df['Return_value'].fillna(0)
final_df['Net_revenue'] = round(final_df['Revenue'] - final_df['Return_value'],2)

In [ ]:
#Resample to monthly data
final_df.set_index('OrderDate' , inplace = True)
monthly = final_df['Net_revenue'].resample('ME').sum()


In [ ]:

#Convert time series
df_ml = monthly.to_frame(name = 'Revenue')


In [ ]:

#Creating Lag Features
for lag in range(1,7):
    df_ml[f'lag_{lag}'] = df_ml['Revenue'].shift(lag)
    
df_ml['time_index'] = np.arange(len(df_ml))

In [ ]:
#Creating Rolling Features
df_ml['rolling_mean_3'] = df_ml['Revenue'].rolling(3).mean()
df_ml['rolling_std_3'] = df_ml['Revenue'].rolling(3).std()


In [ ]:

#Add time based features
df_ml['month'] = df_ml.index.month
df_ml['Quarter'] = df_ml.index.quarter
df_ml['Year'] = df_ml.index.year

#Drop Missing Values
df_ml.dropna(inplace = True)


In [ ]:

#Split Features & Target
split = int(len(df_ml)*0.8)
train = df_ml[:split]
test = df_ml[split : ]

#Train Test Split
x_train = train.drop('Revenue' , axis = 1)
y_train = train['Revenue']
x_test = test.drop('Revenue' , axis = 1)
y_test = test['Revenue']

#Train xgboost (XGBRegressor)
from xgboost import XGBRegressor
model = XGBRegressor(n_estimators = 500 , learning_rate = 0.03 , max_depth = 5 , subsample = 0.8,
                       colsample_bytree = 0.8 , random_state = 42)
model.fit(x_train , y_train)

In [ ]:
#Predict
y_pred = model.predict(x_test)

#Evaluate Model
mae = mean_absolute_error(y_test , y_pred)
mse = mean_squared_error(y_test , y_pred)
rmse = np.sqrt(mse)
print("MAE : " , mae)
print("RMSE : " , rmse)


In [ ]:
#Plot Between Actual V/S Predicted
plt.figure(figsize = (10,5))
plt.plot(y_test.index, y_test , label = 'Actual')
plt.plot(y_test.index , y_pred , label = 'Predicted')
plt.legend()
plt.title("Actual vs Predicted Revenue")
plt.show()

In [ ]:
#Forecast Next 3 months
last_row = df_ml.drop('Revenue' , axis = 1).iloc[-1:].copy()
future_preds = []
for i in range(3):
    pred = model.predict(last_row)[0]
    future_preds.append(pred)

    for lag in range(6,1,-1):
        last_row[f'lag_{lag}'] = last_row[f'lag_{lag-1}']

    last_row['lag_1'] = pred

print("Next 3 months forecast : " , future_preds)
